# 01 - EDA

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns

from sklearn.preprocessing import FunctionTransformer

## Cargar dataset

In [ ]:
df_customer = pd.read_csv("../data/processed/customer_features.csv", index_col="CustomerID")

print(df_customer.shape)
df_customer.head()

## Análisis exploratorio

In [ ]:
df_customer.describe().round(2)

### Comentarios:

* #### **Permanencia :** El 25% de los clientes compran en un solo día. El cliente mediano dura 7 meses y medio. Si hay clientes vitalicios dentro del conjunto de datos analizado.
* #### **Compras :** El cliente con mas compras es 398, mientras el 75% llega hasta 7. La mitad de los clientes compran no más de 3 veces en el periodo de 2 años analizado.
* #### **Canasta promedio :** El cliente con mas unidades promedio por compra es 87167, mientras el 75% llega hasta 258. La mitad de los clientes compran no más de 155 unidades promedio por compra.
* #### **Ticket promedio :** El cliente con mayor gasto promedio por compra es $14,844, mientras el 75% llega hasta $416. La mitad de los clientes gastan no más de $281 promedio por compra.
* #### **Precio promedio :** El cliente que compra en promedio los productos mas caros gasta $10,953 por producto, mientras el 75% llega hasta $2.35. La mitad no gasta más de $1.83 en un producto.
* #### **Precio maximo :** El clinete que compra el producto más caro es de $10,953, mientras el 75% llega hasta $16.9. La mitad no gasta más de $12.7 en su producto más caro.
* #### **Producto distintos :** Los clientes usualmente compran hasta 103 productos distintos en el percentil 75; mientras el que más productos distintos compra tiene 2,550.
* #### **Porcentaje de devoluciones :** El maximo es 400%, la mitad no tienen ninguna devolucion y el 75% llega hasta 28%.

In [ ]:
# Grafica correlacion de pearson y de dispersion en triangulo superior

# variables numericas
df_num = df_customer[[
    "Permanencia",
    "Compras",
    "Canasta_Prom", "Ticket_Prom", "Precio_Prom", "Precio_Max",
    "Productos Distintos"
]]

cmap_corr = mcolors.LinearSegmentedColormap.from_list(
    'custom_corr', ['#ef796d', '#ffffff', '#1164ad']
)

# funcion para anotar la correlacion de pearson en el triangulo superior
def corrfunc(x, y, **kws):
    r = np.corrcoef(x, y)[0, 1]
    ax = plt.gca()

    color_norm = (r + 1) / 2
    ax.set_facecolor(cmap_corr(color_norm))

    ax.annotate(
        f'{r:.2f}',
        xy=(0.5, 0.5),
        xycoords=ax.transAxes,
        ha='center',
        va='center',
        fontsize=12,
        weight='bold',
    )


g = sns.PairGrid(df_num)

# scatter plot en triangulo inferior
g.map_lower(sns.scatterplot, alpha=0.5, s=10)

# histograma/KDE en diagonal
g.map_diag(sns.histplot, kde=True, color='skyblue')

# correlacion en triangulo superior
g.map_upper(corrfunc)

for ax in g.axes.flat:
  if ax is not None:
    xlabel = ax.get_xlabel()
    ylabel = ax.get_ylabel()
    if xlabel:
      ax.set_xlabel(xlabel, fontweight='bold')
    if ylabel:
      ax.set_ylabel(ylabel, fontweight='bold')

plt.tight_layout()
plt.show()

#### Aplicamos transformacion log normal dado que los historgamas no parecen normales y afectaria el reusltados de modelos de densidad

In [ ]:
df_customer_transformed = df_customer.copy()
transformers = dict()

cols_log_normal = [
    "Compras",
    "Canasta_Prom", "Ticket_Prom", "Precio_Prom", "Precio_Max",
    "Productos Distintos"
]
for col in cols_log_normal:
    transformer = FunctionTransformer(func=np.log1p, inverse_func=np.expm1, validate=True)
    transformer.fit(df_customer_transformed[[col]].to_numpy())

    transformers[col] = transformer

for col in cols_log_normal:
    df_customer_transformed[col] = transformers[col].transform(df_customer_transformed[[col]].to_numpy())

df_customer_transformed.describe()

In [ ]:
# Grafica correlacion de pearson y de dispersion en triangulo superior

# variables numericas
df_num = df_customer_transformed[[
    "Permanencia",
    "Compras",
    "Canasta_Prom", "Ticket_Prom", "Precio_Prom", "Precio_Max",
    "Productos Distintos"
]]

cmap_corr = mcolors.LinearSegmentedColormap.from_list(
    'custom_corr', ['#ef796d', '#ffffff', '#1164ad']
)

# funcion para anotar la correlacion de pearson en el triangulo superior
def corrfunc(x, y, **kws):
    r = np.corrcoef(x, y)[0, 1]
    ax = plt.gca()

    color_norm = (r + 1) / 2
    ax.set_facecolor(cmap_corr(color_norm))

    ax.annotate(
        f'{r:.2f}',
        xy=(0.5, 0.5),
        xycoords=ax.transAxes,
        ha='center',
        va='center',
        fontsize=12,
        weight='bold',
    )


g = sns.PairGrid(df_num)

# scatter plot en triangulo inferior
g.map_lower(sns.scatterplot, alpha=0.5, s=10)

# histograma/KDE en diagonal
g.map_diag(sns.histplot, kde=True, color='skyblue')

# correlacion en triangulo superior
g.map_upper(corrfunc)

for ax in g.axes.flat:
  if ax is not None:
    xlabel = ax.get_xlabel()
    ylabel = ax.get_ylabel()
    if xlabel:
      ax.set_xlabel(xlabel, fontweight='bold')
    if ylabel:
      ax.set_ylabel(ylabel, fontweight='bold')

plt.tight_layout()
plt.show()

### Comentarios:

* ### **Correlaciones**
    * #### Permanencia X Compras : Relacion positiva alta de +0.81. A mayor cantidad de tiempo como cliente, mayor la cantidad de compras que se realizan.

In [ ]:
sns.set_theme(style='whitegrid')
fig, axes = plt.subplots(1, 2, figsize=(15, 7))

fig.suptitle(
    'Analisis Exploratorio de Variables Categoricas',
    fontsize=16,
    fontweight='bold',
    y=0.98,  # Ajusta la posición vertical (cerca de 1.0 queda arriba del todo)
)

cat_counts = df_customer_transformed['Pais Principal'].value_counts().sort_values(ascending=False)
cat_counts = pd.concat([
    cat_counts.iloc[:10],
    pd.Series([cat_counts.iloc[10:].sum()],name="count", index=[f"Otros ({cat_counts.iloc[10:].shape[0] :0,.0f})"])
])

sns.barplot(
    x=cat_counts.values,
    y=cat_counts.index,
    ax=axes[0],
    order=cat_counts.index,
    color="#1164ad",
    legend=False,
    width=0.8,
)
axes[0].set_title(
    'Frecuencia Absoluta por Pais', fontweight='bold', fontsize=14
)
axes[0].set_xlabel('Cantidad de Clientes', fontweight='bold', fontsize=11)
axes[0].set_ylabel('Pais Principal', fontweight='bold', fontsize=11)



market_counts = df_customer_transformed["Paises Distintos"].value_counts()
labels = ['1', '2']
colors = ['#1164ad', '#ef796d']

# pctdistance fuera del círculo (ej. 1.2 o 1.25)
wedges, texts, autotexts = axes[1].pie(
    market_counts,
    labels=labels,
    autopct='%1.1f%%',
    startangle=90,
    colors=colors,
    wedgeprops=dict(width=0.5, edgecolor='w'),
    pctdistance=.75,
)

# Estilar textos y porcentajes fuera
for text in texts:
  text.set_fontsize(11)
  text.set_weight('bold')

for autotext in autotexts:
  autotext.set_fontsize(14)
  autotext.set_weight('bold')
  autotext.set_color('#ffffff')

axes[1].set_title(
    'Proporcion de Cantidad de Paises Distintos', fontweight='bold', fontsize=14
)



plt.tight_layout()
plt.show()
sns.reset_defaults()

### Comentarios:

* #### **Pais principal :** El 91% de los clientes hicieron la mayoria de sus compras en UK. El resto se distribuye en los otro 40 paises diferentes.
* #### **Paises Distintos :** El 99.8% de los clientes hicieron todas sus compras en un solo pais. Y el restante, lo hizo en 2.